# sol06: SQL + Python Data Cleaning

Contains:
- the same scenario as `06_mock`
- one complete reference implementation
- grading tests


In [ ]:
import sqlite3
from collections import defaultdict
from datetime import datetime
from typing import Any

RAW_SALES = [
    ("o1", "2025-01-02", "us", "$1,200.00", "2025-01-02T10:00:00"),
    ("o1", "2025-01-02", "US", "$1,250.00", "2025-01-02T12:00:00"),  # latest wins
    ("o2", "01/03/2025", " eu ", "850", "2025-01-03T09:00:00"),
    ("o3", "2025-01-03", "", "N/A", "2025-01-03T09:30:00"),  # invalid amount -> drop
    ("o4", "2025-01-04", "apac", "300.5", "2025-01-04T08:00:00"),
    ("o5", "2025-13-04", "us", "100", "2025-01-04T08:00:00"),  # invalid date -> drop
    ("o6", "2025-01-04", None, " 99.50 ", "2025-01-04T10:00:00"),
]


def make_connection() -> sqlite3.Connection:
    conn = sqlite3.connect(":memory:")
    conn.execute(
        """
        CREATE TABLE sales_raw (
            order_id TEXT NOT NULL,
            order_date TEXT NOT NULL,
            region TEXT,
            amount_text TEXT NOT NULL,
            updated_at TEXT NOT NULL
        )
        """
    )
    conn.executemany(
        "INSERT INTO sales_raw (order_id, order_date, region, amount_text, updated_at) VALUES (?, ?, ?, ?, ?)",
        RAW_SALES,
    )
    conn.commit()
    return conn


In [ ]:
def parse_amount(amount_text: str) -> float | None:
    raw = amount_text.strip().replace("$", "").replace(",", "")
    if raw in {"", "N/A", "n/a", "NULL", "null"}:
        return None
    try:
        return float(raw)
    except ValueError:
        return None


def parse_order_date(raw_date: str) -> str | None:
    for fmt in ("%Y-%m-%d", "%m/%d/%Y"):
        try:
            return datetime.strptime(raw_date, fmt).strftime("%Y-%m-%d")
        except ValueError:
            continue
    return None


def extract_clean_rows(conn: sqlite3.Connection) -> list[dict[str, Any]]:
    sql = """
        SELECT s.order_id, s.order_date, s.region, s.amount_text, s.updated_at
        FROM sales_raw s
        JOIN (
            SELECT order_id, MAX(updated_at) AS max_updated_at
            FROM sales_raw
            GROUP BY order_id
        ) latest
        ON s.order_id = latest.order_id AND s.updated_at = latest.max_updated_at
        ORDER BY s.order_id
    """
    clean: list[dict[str, Any]] = []
    for order_id, order_date, region, amount_text, _updated_at in conn.execute(sql):
        amount = parse_amount(amount_text)
        date_iso = parse_order_date(order_date)
        region_norm = (region or "").strip().upper() or "UNKNOWN"
        if amount is None or date_iso is None or amount <= 0:
            continue
        clean.append(
            {
                "order_id": order_id,
                "order_date": date_iso,
                "region": region_norm,
                "amount": amount,
            }
        )
    return clean


def summarize_by_region(rows: list[dict[str, Any]]) -> dict[str, float]:
    totals: defaultdict[str, float] = defaultdict(float)
    for row in rows:
        totals[row["region"]] += float(row["amount"])
    return {region: round(total, 2) for region, total in sorted(totals.items())}


def top_day(rows: list[dict[str, Any]]) -> tuple[str, float]:
    day_totals: defaultdict[str, float] = defaultdict(float)
    for row in rows:
        day_totals[row["order_date"]] += float(row["amount"])
    if not day_totals:
        raise ValueError("no_rows")
    return sorted(day_totals.items(), key=lambda kv: (-kv[1], kv[0]))[0]


In [ ]:
def run_exam06_tests() -> None:
    conn = make_connection()
    rows = extract_clean_rows(conn)

    assert [row["order_id"] for row in rows] == ["o1", "o2", "o4", "o6"]
    assert rows[0]["amount"] == 1250.0  # latest o1 row wins
    assert rows[1]["order_date"] == "2025-01-03"
    assert rows[3]["region"] == "UNKNOWN"

    summary = summarize_by_region(rows)
    assert summary == {
        "APAC": 300.5,
        "EU": 850.0,
        "UNKNOWN": 99.5,
        "US": 1250.0,
    }

    day, total = top_day(rows)
    assert day == "2025-01-02"
    assert total == 1250.0

    assert parse_amount("N/A") is None
    assert parse_order_date("2025-13-01") is None

    print("06_mock tests passed")


run_exam06_tests()
